In [1]:
#!git clone https://github.com/CharFraza/CPC_ML_tutorial.git
!git clone --branch 2026 https://github.com/CharFraza/CPC_ML_tutorial.git

Cloning into 'CPC_ML_tutorial'...
remote: Enumerating objects: 664, done.
remote: Counting objects: 100% (188/188), done.
remote: Compressing objects: 100% (140/140), done.
remote: Total 664 (delta 107), reused 84 (delta 43), pack-reused 476 (from 1)
Receiving objects: 100% (664/664), 40.43 MiB | 26.71 MiB/s, done.
Resolving deltas: 100% (321/321), done.


## Estimating lifespan normative models
This notebook provides a complete walkthrough of a normative modelling analysis using your own dataset. Training and testing datasets are provided for the purposes of this tutorial. However, you can easily substitute these with your own datasets, provided they follow the same format.

First, if necessary, we install PCNtoolkit.

The latest version of PCNtoolkit includes several updates and new features compared with previous versions. If you have used normative modelling before, we recommend familiarising yourself with the updated syntax and functionality.

For a deeper dive into the theoretical background, see the [protocol update](https://www.biorxiv.org/content/10.64898/2026.02.17.706268v2.full).


In [2]:
# %%
%%capture
!pip install pcntoolkit

In this tutorial, you are going to fit a normative model (or brain growth chart) from scratch. Usually, we do not recommend doing this unless you have the computational resources needed to analyse 100,000 images. In the second tutorial, we will show you a much more appropriate technique using federated learning, so that you can easily apply the models to your own dataset. But let's get started!

### Some household notes
* Make sure you read all the text. It is easy to click and play, but answering the questions is where the real learning happens.
* Try playing around with different parameter settings and variables to make sure you really understand what is happening.
* If you are an advanced user, consider adapting the methods to your own data. You can download the tutorials locally to avoid having your data on a Google server ;)


In [3]:
# Import all the required libraries
import os
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

from pcntoolkit import (
    BLR,
    BsplineBasisFunction,
    NormativeModel,
    NormData,
    plot_centiles,
    plot_centiles_advanced,
    plot_qq,
    plot_ridge,
)

Now, we configure the locations in which the data are stored.

**Notes:**
- The data are assumed to be in .CSV format and will be loaded as pandas dataframes
- Generally the raw data will be in a different location to the analysis
- The data can have several columns/covariates but some are required by the script, i.e. 'age', 'sex' and 'site', plus the phenotypes you wish to estimate

In [9]:
# where the raw data are stored

# where the analysis takes place
root_dir = '/content/CPC_ML_tutorial'
!cd '/content/CPC_ML_tutorial'

data_dir = os.path.join(root_dir, 'data')

out_dir = os.path.join(root_dir, 'models', 'test')

# create the output directory if it does not already exist
os.makedirs(out_dir, exist_ok=True)

## Load the data
Now we load the data.
We will load one pandas dataframe for the training set and one dataframe
for the test set. We also configure a list of site ids.


In [10]:
df_tr = pd.read_csv(
    os.path.join(data_dir, 'train_data.csv'),
    index_col=0
)

df_te = pd.read_csv(
    os.path.join(data_dir, 'test_data.csv'),
    index_col=0
)

# extract a list of unique site ids from the training set
site_ids = sorted(set(df_tr['site'].to_list()))

print(site_ids)

['cam', 'hcp', 'ixi']


### Configure which models to fit

Next, we load the image derived phenotypes (IDPs) which we will process in this analysis. This is effectively just a list of columns in your dataframe. Here we estimate normative models for the left hemisphere, right hemisphere and cortical structures.

In [ ]:
# we choose here to process all idps. Uncomment lines 2-7 (and comment line 11) to run models for the whole brain, but we suggest just starting with several ROIs
#os.chdir(root_dir)
#!wget -nc https://raw.githubusercontent.com/CharFraza/CPC_ML_tutorial/master/data/task1_phenotypes.txt
#with open(os.path.join(root_dir,'task1_phenotypes.txt')) as f:
#  idp_ids = f.read().splitlines()
#for idx, ele in enumerate(idp_ids):
#        idp_ids[idx] = ele.replace('\t', '')

# we could also just specify a list of IDPs. Use this line to run just 2 models (not the whole brain)...this is a good place to start. If you have time,
# you can uncomment the above line and run the whole brain models. Be sure to comment out this line if you uncomment the above line.
idp_ids = ['lh_MeanThickness_thickness', 'rh_MeanThickness_thickness']

In [ ]:
# Pandas can give us a quick summary of the numerical variables,
# including the number of observations, mean, standard deviation,
# minimum and maximum values.
# Make sure you understand what each variable represents in the context of your study!
df_tr.describe()
df_te.describe()

In [ ]:
# Explore the data a little further and make sure you understand what each variable represents.
# Compare the values also to df_te to see if the distributions are similar between the training and test sets.
display(df_tr.head())

fig, ax = plt.subplots(2, 2, figsize=(12, 8))

sns.histplot(df_tr, x='age', bins=20, ax=ax[0, 0])
ax[0, 0].set_title('Age distribution')

sns.countplot(
    data=df_tr,
    y='site',
    order=df_tr['site'].value_counts().index,
    ax=ax[0, 1]
)
ax[0, 1].set_title('Subjects by site')

sns.countplot(data=df_tr, x='sex', ax=ax[1, 0])
ax[1, 0].set_title('Subjects by sex')

sns.scatterplot(
    data=df_tr,
    x='age',
    y=idp_ids[0], # Change this to the IDP you want to plot
    hue='site',
    style='sex',
    legend=False,
    ax=ax[1, 1]
)

plt.tight_layout()
plt.show()

### Covariates, batch effects and response variable selection
In general, the choice of covariates and batch effects depends on your research question. However, age, sex, and site are commonly recommended starting points for normative modelling:
- Batch effects allow your model to account for systematic differences between groups or participants. For example, when people are scanned at different sites or using different scanners, differences may appear in the data because of the site or scanner rather than genuine biological differences. Batch effects allow the model to account for these systematic differences. They can be used for variables such as site, scanner, sex, ancestry, or imaging field strength, depending on the application. For a more detailed explanation of batch effects and partial pooling in normative modelling, see the [protocol update](https://www.biorxiv.org/content/10.64898/2026.02.17.706268v2.full).
- Covariates are variables that the model uses to explain variation in the response variable. They are typically continuous variables, such as age or height. In normative modelling, covariates are often variables that we expect to be related to the biological measure we are modelling. For example, a model might predict brain volume (the response variable) based on age (the covariate), while accounting for differences between sex and scanning sites using batch effects.

Note: If you are interested in understanding how sex affects the outcome, it may make more sense to treat sex as a covariate rather than simply as a batch effect :)

In [ ]:
cols_cov = ['age']
batch_effects = ['sex', 'site']

## Configure model parameters
We now configure parameters for the regression model used to fit the normative model. Specifically, we use a warped Bayesian linear regression model, applying a SinhArcsinh warp to handle non-Gaussianity. To capture non-linearity, we use the default cubic B-spline basis with five knot points, which requires no additional parameters. However, we do specify age range limits, padding a few years beyond the observed range. Finally, we set a few options that govern the model estimation process.
or further details about the likelihood warping approach, see [Fraza et al 2021](https://www.sciencedirect.com/science/article/pii/S1053811921009873?via%3Dihub).
For more details on B-spline expansions, see: https://www.geeksforgeeks.org/data-analysis/b-splines-using-scipy/

In [ ]:
# which warping function to use? We can set this to None in order to fit a vanilla Gaussian noise model
use_warp = True
wrap_function = 'warpsinharcsinh'

basis_function = BsplineBasisFunction(
    degree=3,
    nknots=5
)

# Do we want to force the model to be refit every time?
# When training normative model from scratch like we are doing in this notebook (not re-using a pre-trained model),
# this variable should be = True
force_refit = True

# Absolute Z treshold above which a sample is considered to be an outlier (without fitting any model)
outlier_thresh = 7

# Data preparation
In the PCNtoolkit the NormData class is used for general data cleaning, harmonization, and preparation.
Check some of the outputs carefully, are the batch_effects set up right, what are your response variables? Etc.

In [ ]:
response_vars = idp_ids

norm_train = NormData.from_dataframe(
    name='train',
    dataframe=df_tr,
    covariates=cols_cov,
    batch_effects=batch_effects,
    response_vars=response_vars
)

norm_test = NormData.from_dataframe(
    name='test',
    dataframe=df_te,
    covariates=cols_cov,
    batch_effects=batch_effects,
    response_vars=response_vars
)

print(norm_train)
print(norm_test)

## Model settup
Here we define the settings for our BLR normative model, using the B-spline for non-linear relationships, allowing variance to change with age (heteroskedastic) and setting the fixed effects to account for batch_effects, allowing differences between sites/groups to influence the mean and variance of the model.
Finally a sinh-arcsinh warping is used to handle non-Gaussian residuals.
Play around with the settings and see how your model changes~

In [ ]:
blr_kwargs = {
    'name': 'lifespan_blr',

    # cubic B-spline basis with 5 knots as selected before
    'basis_function_mean': basis_function,

    # allow the variance to depend on the covariates, in this case age
    'heteroskedastic': True,

    # model the batch effects
    'fixed_effect': True,
    'fixed_effect_slope': True,
    'fixed_effect_var_slope': True,
}

# optionally add Sinh-Arcsinh warping to adress non-Gaussian noise in the data. This is a flexible warping function that can model skewness and heavy tails in the distribution of the residuals.
if use_warp:
    blr_kwargs['warp_name'] = wrap_function

# %%
blr = BLR(
    **blr_kwargs
)

model = NormativeModel(
    template_regression_model=blr,
    savemodel=True,
    evaluate_model=True,
    saveresults=True,
    saveplots=True,
    save_dir=out_dir,
    inscaler='standardize',
    outscaler='standardize',
)

### Fit the models

Now we fit the models. This involves looping over the IDPs we have selected.

In [ ]:
model.fit_predict(
    norm_train,
    norm_test
)

## Explore and visualize the normative model
We can now get to the fun part of visualizing our normative models, using centile curves and QQ-plots.

In [ ]:
# These centile plots show the expected distribution of the response variable across the range of the covariate (e.g., age), similar to a growth chart.
# The shaded areas represent the 5th, 25th, 50th, 75th, and 95th percentiles of the expected distribution,
# while the scatter points represent the actual data points from the test set.
# This allows us to visually assess how well the model captures the variability in the data and whether there are any systematic deviations from the expected distribution.
plot_centiles(
    model,
    scatter_data=norm_test
)

In [ ]:
# Checking model fit with a Q–Q plot
# A Q–Q plot (quantile-quantile plot) is a graphical tool to assess if a dataset follows a particular theoretical distribution,
# in this case, the normal distribution, see if the line follows the 45-degree line,
# which would indicate that the data follows a normal distribution.
plot_qq(
    norm_test,
    plot_id_line=True
)

### Compute error metrics

In this section we show the following error metrics for all IDPs (all evaluated on the test set): assess the goodness of fit between the predicted probabilities of a model and the actual observed outcomes.
- Negative log likelihood (NLL): NLL assesses the goodness of fit between the predicted probabilities of a model and the actual observed outcomes. In this case, it measures the discrepancy between the predicted probabilities of the model for the IDPs (Independent Data Points) and the actual outcomes on the test set.
- Explained variance (EV): EV assesses how much of the total variation in the dependent variable (IDP) is explained by the independent variables. In the context of this analysis, it quantifies the extent to which the independent variables account for the variability observed in the IDPs on the test set.
- Mean standardized log loss (MSLL): MSLL takes into account both the mean error and the estimated prediction variance. It is used to evaluate the performance of the model, and in this case, a lower MSLL indicates a better-fitting model for the IDPs on the test set.
- Bayesian information criteria (BIC): BIC is a model selection criterion that balances the goodness of fit to the data with the model's complexity. It penalizes models with higher flexibility and aims to find the best trade-off. Lower BIC scores indicate models that better explain the IDPs on the test set while considering the model complexity.
- Skew and Kurtosis of the Z-distribution: Skewness and kurtosis are statistical measures used to assess the shape and characteristics of a distribution. They provide information about how well the warping function performed in terms of capturing the departure from a normal distribution for the IDPs.

For more information on the different model selection criteria see [Fraza et al 2021](https://www.biorxiv.org/content/10.1101/2021.04.05.438429v1)

In [ ]:
# Evaluating model performance
train_metrics = norm_train.get_statistics_df()
test_metrics = norm_test.get_statistics_df()


# We can also look at a set of summary statistics to evaluate how well the normative model performs on both the training and test datasets.
print("Training set metrics:")
display(train_metrics)


#
print("Test set metrics:")
display(test_metrics)

In [ ]:
for idp in idp_ids:

    z = norm_test['Z'].sel(
        response_vars=idp
    ).values

    plt.figure(
        figsize=(8, 6)
    )

    plt.hist(
        z,
        bins=30
    )

    plt.xlabel('Z-score')
    plt.ylabel('Count')

    plt.title(
        f'Z-score distribution: {idp}'
    )

    plt.show()


In [ ]:
plot_qq(
    norm_test,
    plot_id_line=True,
    hue_data='sex',
    split_data='sex'
)

In [ ]:
plot_ridge(
    norm_test,
    'Z',
    split_by='sex'
)

## Questions to discuss
1. Model selection: Which model selection criteria would you use to choose the optimal model?
2. Model flexibility: What happens when you change the warping or B-spline settings?
3. Bias-variance tradeoff: How would you consider the bias-variance tradeoff when deciding the models parameters?
4. Which independent variables do you think are important to add to the normative model?
5. Are there other model selection criteria that you think should be considered?

## Suggested further readings

1. [PCNtoolkit Background](https://pcntoolkit.readthedocs.io/en/latest/pages/pcntoolkit_background.html)
2. [Conceptualizing mental disorders as deviations from normative functioning](https://www.nature.com/articles/s41380-019-0441-1)
3. [Understanding Heterogeneity in Clinical Cohorts Using Normative Models: Beyond Case-Control Studies
](https://www.sciencedirect.com/science/article/pii/S0006322316000020)